#Langues turciques

In [ ]:
# Example: Generate embeddings from your idiom sentences
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [ ]:
import json # Added this line

# Liste des fichiers à fusionner
files = ['azerbaijani.json', 'turkish.json', 'uzbek.json']
all_data = []

for filename in files:
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
            # Vérifier si c'est une liste (format standard de tes fichiers)
            if isinstance(data, list):
                for entry in data:
                    row = {
                        'contains_idioms': entry.get('idiom', ''),
                        'original_text': entry.get('example', ''),
                        'text': entry.get('figurative_meaning', ''),
                        'filename': filename  # Ajoute le nom du fichier source
                    }
                    all_data.append(row)
    except Exception as e:
        print(f"Erreur de lecture pour {filename}: {e}")

# Création du DataFrame et sélection des colonnes
df = pd.DataFrame(all_data)
df = df[['contains_idioms', 'text', 'original_text', 'filename']]

# Sauvegarde en CSV
df.to_csv('turcique.csv', index=False)

print(f"Fichier 'turcique.csv' créé ({len(df)} lignes)")

Fichier 'turcique.csv' créé (29 lignes)


In [ ]:
model_name = "xlm-roberta-base"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
except Exception as e:
    raise RuntimeError(f"Failed to load model '{model_name}': {e}")

# Locate data file (support .csv and .parquet)
cwd = Path().resolve()
candidates = [
    cwd / "turcique.csv",
    cwd / "turcique.parquet",
    cwd / "turcique"
]
csv_path = None
for p in candidates:
    if p.exists():
        csv_path = p
        break
# fallback: find files starting with base name
if csv_path is None:
    for f in cwd.iterdir():
        if f.name.startswith("turcique") and f.suffix in {".csv", ".parquet", ""}:
            csv_path = f
            break
if csv_path is None:
    # helpful error with nearby files
    nearby = [p.name for p in cwd.glob("turcique*")]
    raise FileNotFoundError(f"Couldn't find 'turcique' (.csv or .parquet) in {cwd}. Candidates found: {nearby}")

# load into pandas
if str(csv_path).lower().endswith(".parquet"):
    df = pd.read_parquet(csv_path)
else:
    df = pd.read_csv(csv_path)

print(f"Dataset chargé: {len(df)} lignes — fichier: {csv_path}")
print(df.head())

# Extract text column (allow common alternative column names)
text_col = None
for c in ["text", "sentence", "sent", "original_text"]:
    if c in df.columns:
        text_col = c
        break
if text_col is None:
    raise KeyError("Le fichier doit contenir une colonne 'text' (ou 'sentence','sent','original_text').")
if text_col != "text":
    df = df.rename(columns={text_col: "text"})
sentences = df["text"].astype(str).tolist()
contains_idioms = df["contains_idioms"].tolist() if "contains_idioms" in df.columns else [False] * len(sentences)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Dataset chargé: 29 lignes — fichier: /content/turcique.csv
                   contains_idioms  \
0                    İki ucu kəsik   
1          Düşmənini dostunla tanı   
2               Göz var, nizam var   
3    Söz gümüşdür, susmaq qızıldır   
4  Bir əl ilə alov yandırmaq olmaz   

                                                text  \
0                  A situation with no good options.   
1                To be cautious about whom to trust.   
2         Things should be done properly or orderly.   
3  Sometimes it's better to remain silent than to...   
4                 Teamwork is essential for success.   

                                       original_text          filename  
0  Choosing between those two jobs is like being ...  azerbaijani.json  
1  He learned to düşmənini dostunla tanı after be...  azerbaijani.json  
2  In this project, we need to remember that göz ...  azerbaijani.json  
3  In that situation, remember that söz gümüşdür,...  azerbaijani.json  
4  To achi

In [ ]:
# Générer les embeddings (batched, GPU, mixed precision)
import math
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

use_fp16 = device.type == "cuda"

if use_fp16:
    model.half()  # reduce memory & speed up on GPU

def mean_pooling(outputs, attention_mask):
    token_embeddings = outputs.last_hidden_state  # (batch, seq_len, dim)
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

batch_size = 64  # try 32/64/128 depending on GPU memory
n = len(sentences)
embeddings_batches = []

print("Génération des embeddings (batched)...")
for start in tqdm(range(0, n, batch_size)):
    batch = sentences[start : start + batch_size]
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        if use_fp16:
            with torch.cuda.amp.autocast():
                outputs = model(**inputs)
        else:
            outputs = model(**inputs)

        batch_emb = mean_pooling(outputs, inputs["attention_mask"])  # (batch, dim)
        embeddings_batches.append(batch_emb.cpu().numpy())

embeddings = np.vstack(embeddings_batches)



Génération des embeddings (batched)...


100%|██████████| 1/1 [00:01<00:00,  1.57s/it]


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv('turcique.csv')

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(df['text'].tolist(), show_progress_bar=True)

reducer = UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
projection = reducer.fit_transform(embeddings)

df['projection_x'] = projection[:, 0]
df['projection_y'] = projection[:, 1]

nn = NearestNeighbors(n_neighbors=16, metric='cosine')
nn.fit(embeddings)
distances, indices = nn.kneighbors(embeddings)

df['neighbors'] = [indices[i][1:].tolist() for i in range(len(df))]

# Sauvegarder tout
df.to_parquet('embeddings_ready.parquet')
print("Données sauvegardées dans embeddings_ready.parquet")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Données sauvegardées dans embeddings_ready.parquet


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Charger les données depuis le fichier parquet
df_graph = pd.read_parquet('embeddings_ready.parquet')

# Créer la figure Plotly
fig = go.Figure()

# Ajouter les lignes de connexion entre les voisins
# Cela crée un maillage de graph entre les points projetés.
for i, row in df_graph.iterrows():
    current_x = row['projection_x']
    current_y = row['projection_y']

    # Récupérer les indices des voisins
    # Assurez-vous que la colonne 'neighbors' contient une liste d'indices (entiers)
    neighbors_indices = row['neighbors']

    for neighbor_idx in neighbors_indices:
        if neighbor_idx < len(df_graph): # Vérification pour éviter les erreurs d'index
            neighbor_x = df_graph.loc[neighbor_idx, 'projection_x']
            neighbor_y = df_graph.loc[neighbor_idx, 'projection_y']

            fig.add_trace(go.Scatter(
                x=[current_x, neighbor_x],
                y=[current_y, neighbor_y],
                mode='lines',
                line=dict(width=0.4, color='rgba(150, 150, 150, 0.1)'), # Rendre les lignes plus subtiles
                hoverinfo='none', # Pas d'info-bulle pour les lignes
                showlegend=False
            ))

# Ajouter les points du scatter plot (par-dessus les lignes), colorés par 'filename'
# Utilisation de plotly.express pour une coloration automatique et une légende facile
fig_points = px.scatter(
    df_graph,
    x='projection_x',
    y='projection_y',
    color='filename', # Couleur basée sur le nom du fichier
    hover_name='contains_idioms',
    hover_data={
        'contains_idioms': True,
        'text': True,
        'original_text': True,
        'filename': True,
        'projection_x': False, # Ne pas afficher dans le hover_data
        'projection_y': False  # Ne pas afficher dans le hover_data
    }
)

# Ajouter les traces de points colorés à la figure principale
for trace in fig_points.data:
    fig.add_trace(trace)

# Mettre à jour la mise en page du graphique
fig.update_layout(
    title='UMAP Projection of Idioms with Nearest Neighbors (Colored by Source File)',
    xaxis_title='Projection X',
    yaxis_title='Projection Y',
    hovermode='closest',
    showlegend=True, # Assurez-vous que la légende est visible
    height=700 # Augmenter la hauteur pour une meilleure visibilité
)

# Afficher le graphique
fig.show()